# Semana 2 - Planificacion de modelado y split train/test

Este notebook prepara la etapa de modelado sin entrenar modelos todavia. El objetivo de esta parte es dejar definido el experimento: que se quiere predecir, que variables entran como candidatas, que riesgos metodologicos vienen del EDA y como se separan los datos de forma reproducible.

La consigna pide usar un split estratificado porque churn es una clase minoritaria. Si se separa train/test sin cuidar la proporcion de clases, el modelo podria aprender con una distribucion y evaluarse con otra.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_PATH = Path("E Commerce Dataset.xlsx - E Comm.csv")
RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "Churn"
ID_COL = "CustomerID"

df_raw = pd.read_csv(DATA_PATH)
missing_cols = df_raw.columns[df_raw.isna().any()].tolist()

df_raw.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.000,Mobile Phone,3,6.000,Debit Card,Female,3.000,3,Laptop & Accessory,2,Single,9,1,11.000,1.000,1.000,5.000,160
1,50002,1,NaN,Phone,1,8.000,UPI,Male,3.000,4,Mobile,3,Single,7,1,15.000,0.000,1.000,0.000,121
2,50003,1,NaN,Phone,1,30.000,Debit Card,Male,2.000,4,Mobile,3,Single,6,1,14.000,0.000,1.000,3.000,120
3,50004,1,0.000,Phone,3,15.000,Debit Card,Male,2.000,4,Laptop & Accessory,5,Single,8,0,23.000,0.000,1.000,3.000,134
4,50005,1,0.000,Phone,1,12.000,CC,Male,NaN,3,Mobile,5,Single,3,0,11.000,1.000,1.000,3.000,130


## 1. Definicion de X e y

`CustomerID` no entra como predictor porque es un identificador, no una senal de comportamiento. `Churn` se separa como variable objetivo.

Todavia no se imputan nulos ni se codifican categorias. Esas transformaciones tienen que vivir en el pipeline de entrenamiento para evitar fuga de informacion desde test hacia train.

In [2]:
feature_cols = [col for col in df_raw.columns if col not in [ID_COL, TARGET]]

X = df_raw[feature_cols].copy()
y = df_raw[TARGET].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Churn rate total: {y.mean() * 100:.2f}%")

X.head()

X shape: (5630, 18)
y shape: (5630,)
Churn rate total: 16.84%


,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,4.000,Mobile Phone,3,6.000,Debit Card,Female,3.000,3,Laptop & Accessory,2,Single,9,1,11.000,1.000,1.000,5.000,160
1,NaN,Phone,1,8.000,UPI,Male,3.000,4,Mobile,3,Single,7,1,15.000,0.000,1.000,0.000,121
2,NaN,Phone,1,30.000,Debit Card,Male,2.000,4,Mobile,3,Single,6,1,14.000,0.000,1.000,3.000,120
3,0.000,Phone,3,15.000,Debit Card,Male,2.000,4,Laptop & Accessory,5,Single,8,0,23.000,0.000,1.000,3.000,134
4,0.000,Phone,1,12.000,CC,Male,NaN,3,Mobile,5,Single,3,0,11.000,1.000,1.000,3.000,130


## 2. Split train/test estratificado

Se usa 80% para entrenamiento y 20% para test, con `stratify=y` para conservar la proporcion de clientes que churnean en ambos conjuntos. El `random_state` fijo hace que el split sea reproducible.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame({
    "dataset": ["total", "train", "test"],
    "filas": [len(y), len(y_train), len(y_test)],
    "clientes_churn": [int(y.sum()), int(y_train.sum()), int(y_test.sum())],
    "clientes_no_churn": [int((y == 0).sum()), int((y_train == 0).sum()), int((y_test == 0).sum())],
    "tasa_churn_%": [round(y.mean() * 100, 2), round(y_train.mean() * 100, 2), round(y_test.mean() * 100, 2)],
})

split_summary

,dataset,filas,clientes_churn,clientes_no_churn,tasa_churn_%
0,total,5630,948,4682,16.840
1,train,4504,758,3746,16.830
2,test,1126,190,936,16.870


## 3. Validaciones del split

Estas validaciones confirman que no se perdieron filas, que train y test no se pisan y que la tasa de churn quedo practicamente igual en ambos conjuntos.

In [4]:
train_ids = set(df_raw.loc[X_train.index, ID_COL])
test_ids = set(df_raw.loc[X_test.index, ID_COL])

assert len(X_train) + len(X_test) == len(df_raw), "Train + test no suma el total de filas"
assert len(train_ids.intersection(test_ids)) == 0, "Hay clientes repetidos entre train y test"
assert X_train.index.intersection(X_test.index).empty, "Hay indices repetidos entre train y test"
assert y_train.sum() + y_test.sum() == y.sum(), "La cantidad de churn no se conserva"

max_rate_diff_pp = abs(y_train.mean() - y_test.mean()) * 100
assert max_rate_diff_pp < 0.25, f"Diferencia de churn train/test demasiado alta: {max_rate_diff_pp:.3f} pp"

print("Validaciones OK")
print(f"Filas train: {len(X_train):,}")
print(f"Filas test: {len(X_test):,}")
print(f"Churn train: {y_train.mean() * 100:.2f}%")
print(f"Churn test: {y_test.mean() * 100:.2f}%")
print(f"Diferencia train/test: {max_rate_diff_pp:.3f} puntos porcentuales")

Validaciones OK
Filas train: 4,504
Filas test: 1,126
Churn train: 16.83%
Churn test: 16.87%
Diferencia train/test: 0.044 puntos porcentuales


## 4. Imputacion planificada despues del split

La imputacion del EDA sirvio para analizar sin perder filas. Para modelado se recalcula despues del split para evitar leakage.

Criterio acordado:

- `WarehouseToHome` y `CouponUsed`: imputacion por mediana calculada solo con train.
- `Tenure`, `HourSpendOnApp`, `OrderAmountHikeFromlastYear`, `OrderCount` y `DaySinceLastOrder`: comparar KNN vs regresion iterativa usando solo train y elegir el menor MAE por variable.
- Test se transforma con los parametros aprendidos en train.

In [5]:
median_imputation_cols = ["WarehouseToHome", "CouponUsed"]
model_imputation_cols = [
    "Tenure",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "OrderCount",
    "DaySinceLastOrder",
]

numeric_imputation_features = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CityTier",
    "NumberOfDeviceRegistered",
    "SatisfactionScore",
    "NumberOfAddress",
    "Complain",
    "CashbackAmount",
]

train_medians = X_train[median_imputation_cols].median()

X_train_imputation_base = X_train[numeric_imputation_features].copy()
X_test_imputation_base = X_test[numeric_imputation_features].copy()

for col in median_imputation_cols:
    X_train_imputation_base[col] = X_train_imputation_base[col].fillna(train_medians[col])
    X_test_imputation_base[col] = X_test_imputation_base[col].fillna(train_medians[col])

pd.DataFrame({
    "variable": median_imputation_cols,
    "mediana_train": [train_medians[col] for col in median_imputation_cols],
    "nulos_train_antes": [int(X_train[col].isna().sum()) for col in median_imputation_cols],
    "nulos_test_antes": [int(X_test[col].isna().sum()) for col in median_imputation_cols],
})

,variable,mediana_train,nulos_train_antes,nulos_test_antes
0,WarehouseToHome,14.000,206,45
1,CouponUsed,1.000,208,48


In [6]:
def fit_scaled_imputer(data, imputer):
    """Ajusta scaler e imputador sobre train y devuelve ambos objetos junto con la matriz imputada."""
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(data),
        columns=data.columns,
        index=data.index,
    )
    X_imputed_scaled = imputer.fit_transform(X_scaled)
    X_imputed = pd.DataFrame(
        scaler.inverse_transform(X_imputed_scaled),
        columns=data.columns,
        index=data.index,
    )
    return scaler, imputer, X_imputed


def transform_scaled_imputer(data, scaler, imputer):
    X_scaled = pd.DataFrame(
        scaler.transform(data),
        columns=data.columns,
        index=data.index,
    )
    X_imputed_scaled = imputer.transform(X_scaled)
    return pd.DataFrame(
        scaler.inverse_transform(X_imputed_scaled),
        columns=data.columns,
        index=data.index,
    )


def evaluate_scaled_imputer_for_column(data, target_col, imputer, random_state=RANDOM_STATE, mask_fraction=0.20):
    observed_idx = data.index[data[target_col].notna()].to_numpy()
    rng = pd.Series(observed_idx).sample(
        n=max(1, int(len(observed_idx) * mask_fraction)),
        random_state=random_state,
    ).to_numpy()

    X_masked = data.copy()
    y_true = X_masked.loc[rng, target_col].copy()
    X_masked.loc[rng, target_col] = pd.NA

    _, _, X_imputed = fit_scaled_imputer(X_masked, imputer)
    y_pred = X_imputed.loc[rng, target_col]
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "n_validacion": len(rng),
    }


imputer_specs = {
    "knn_k5": KNNImputer(n_neighbors=5, weights="distance"),
    "iterative_regression": IterativeImputer(
        random_state=RANDOM_STATE,
        max_iter=20,
        initial_strategy="median",
        sample_posterior=False,
    ),
}

imputation_results = []
for target_col in model_imputation_cols:
    for method, imputer in imputer_specs.items():
        metrics = evaluate_scaled_imputer_for_column(
            X_train_imputation_base,
            target_col=target_col,
            imputer=imputer,
        )
        imputation_results.append({
            "variable": target_col,
            "metodo": method,
            **metrics,
        })

imputation_results = pd.DataFrame(imputation_results)
best_imputer_by_variable = imputation_results.loc[
    imputation_results.groupby("variable")["MAE"].idxmin()
].sort_values("variable")

imputation_results.sort_values(["variable", "MAE"])

,variable,metodo,MAE,RMSE,n_validacion
8,DaySinceLastOrder,knn_k5,1.618,2.472,851
9,DaySinceLastOrder,iterative_regression,2.324,2.946,851
2,HourSpendOnApp,knn_k5,0.427,0.587,861
3,HourSpendOnApp,iterative_regression,0.540,0.667,861
4,OrderAmountHikeFromlastYear,knn_k5,2.090,3.013,859
5,OrderAmountHikeFromlastYear,iterative_regression,2.992,3.586,859
6,OrderCount,knn_k5,0.846,1.653,858
7,OrderCount,iterative_regression,1.276,2.010,858
0,Tenure,knn_k5,4.264,6.322,858
1,Tenure,iterative_regression,6.064,7.737,858


In [7]:
fitted_imputers = {}

for method, imputer in imputer_specs.items():
    scaler, fitted_imputer, X_train_imputed_numeric = fit_scaled_imputer(
        X_train_imputation_base,
        imputer,
    )
    X_test_imputed_numeric = transform_scaled_imputer(
        X_test_imputation_base,
        scaler,
        fitted_imputer,
    )
    fitted_imputers[method] = {
        "scaler": scaler,
        "imputer": fitted_imputer,
        "train_numeric": X_train_imputed_numeric,
        "test_numeric": X_test_imputed_numeric,
    }

X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

for col in median_imputation_cols:
    X_train_imputed[col] = X_train_imputed[col].fillna(train_medians[col])
    X_test_imputed[col] = X_test_imputed[col].fillna(train_medians[col])

for _, row in best_imputer_by_variable.iterrows():
    col = row["variable"]
    method = row["metodo"]
    train_missing_mask = X_train_imputed[col].isna()
    test_missing_mask = X_test_imputed[col].isna()
    X_train_imputed.loc[train_missing_mask, col] = fitted_imputers[method]["train_numeric"].loc[train_missing_mask, col]
    X_test_imputed.loc[test_missing_mask, col] = fitted_imputers[method]["test_numeric"].loc[test_missing_mask, col]

imputation_check = pd.DataFrame({
    "dataset": ["X_train original", "X_train imputado", "X_test original", "X_test imputado"],
    "nulos_totales": [
        int(X_train.isna().sum().sum()),
        int(X_train_imputed.isna().sum().sum()),
        int(X_test.isna().sum().sum()),
        int(X_test_imputed.isna().sum().sum()),
    ],
})

display(best_imputer_by_variable)
imputation_check

,variable,metodo,MAE,RMSE,n_validacion
8,DaySinceLastOrder,knn_k5,1.618,2.472,851
2,HourSpendOnApp,knn_k5,0.427,0.587,861
4,OrderAmountHikeFromlastYear,knn_k5,2.090,3.013,859
6,OrderCount,knn_k5,0.846,1.653,858
0,Tenure,knn_k5,4.264,6.322,858


,dataset,nulos_totales
0,X_train original,1491
1,X_train imputado,0
2,X_test original,365
3,X_test imputado,0


## 5. Validaciones de imputacion

Estas validaciones revisan que la imputacion elimine los nulos numericos esperados y que no modifique el target ni la cantidad de filas. Las variables categoricas no tenian nulos en el dataset original.

In [8]:
assert X_train_imputed.shape == X_train.shape
assert X_test_imputed.shape == X_test.shape
assert y_train.equals(y.loc[X_train.index])
assert y_test.equals(y.loc[X_test.index])
assert X_train_imputed[missing_cols].isna().sum().sum() == 0
assert X_test_imputed[missing_cols].isna().sum().sum() == 0

print("Validaciones de imputacion OK")
print(f"Nulos originales en train: {int(X_train[missing_cols].isna().sum().sum()):,}")
print(f"Nulos imputados en train: {int(X_train_imputed[missing_cols].isna().sum().sum()):,}")
print(f"Nulos originales en test: {int(X_test[missing_cols].isna().sum().sum()):,}")
print(f"Nulos imputados en test: {int(X_test_imputed[missing_cols].isna().sum().sum()):,}")

Validaciones de imputacion OK
Nulos originales en train: 1,491
Nulos imputados en train: 0
Nulos originales en test: 365
Nulos imputados en test: 0


## 6. Proxima etapa

Con el split congelado, el siguiente notebook o la siguiente seccion deberia construir pipelines de preprocesamiento y modelos. Para evitar leakage:

- Ajustar imputadores solo con `X_train`.
- Ajustar encoders solo con `X_train`.
- Ajustar escaladores solo con `X_train` cuando el modelo lo necesite.
- Transformar `X_test` usando exclusivamente objetos ya ajustados en train.
- Comparar modelos con metricas adecuadas para desbalance: recall, precision, F1, PR-AUC y matriz de confusion.

Decision pendiente antes de entrenar: confirmar si `Complain` y `DaySinceLastOrder` estan disponibles antes del momento de prediccion. Si no lo estan, pueden introducir leakage temporal.